In [ ]:
import pandas as pd
import numpy as np
import math
import warnings
import plotly.express as px
# Chia thành 2 biểu đồ trong cùng một figure
from plotly.subplots import make_subplots
import seaborn as sns
warnings.filterwarnings('ignore')

DATA_DIR  = '../data/raw/'

## **Part 1. Descriptive — "What happened?**

Các bảng dữ liệu sử dụng: sales.csv, orders.csv, products.csv, customers.csv

**1. Doanh thu tổng quan**
- Revenue theo ngày/tháng/năm
- Gross profit margin
- COGS vs Revenue trend
- Biểu đồ sử dụng: Line chart + area chart

**2. Sản phẩm & khách hàng**
- Top sản phẩm / category
- Phân bố segment, size
- Số KH theo age_group
- Biểu đồ sử dụng: Bar chart + pie chart

**3. Vận hành**
- Tỷ lệ trả hàng / huỷ đơn
- Web traffic theo source
- Rating phân bố 1–5 sao
- Biểu đồ sử dụng: Histogram + heatmap

In [ ]:
# 1. Doanh thu tổng quan
# Revenue theo ngày tháng năm
sales_df = pd.read_csv(DATA_DIR + 'sales.csv')
sales_df['Date'] = pd.to_datetime(sales_df['Date'])
total_revenue = sales_df['Revenue'].sum()

# vẽ biểu đồ doanh thu theo thời gian bằng plotly
daily_revenue = sales_df.groupby('Date')['Revenue'].sum().reset_index()
fig = px.line(daily_revenue, x='Date', y='Revenue', title='Doanh thu theo ngày')
fig.update_layout(xaxis_title='Ngày', yaxis_title='Doanh thu (VND)')
fig.show()

# Gross profit margin theo ngày tháng năm (lấy theo median để giảm ảnh hưởng của outliers)
sales_df['Gross Profit Margin'] = (sales_df['Revenue'] - sales_df['COGS']) / sales_df['Revenue']
daily_gpm = sales_df.groupby('Date')['Gross Profit Margin'].median().reset_index()
fig = px.line(daily_gpm, x='Date', y='Gross Profit Margin', title='Gross Profit Margin theo ngày')
fig.update_layout(xaxis_title='Ngày', yaxis_title='Gross Profit Margin')
fig.show()

# COGS and Revenue trend (area chart)
daily_cogs_revenue = sales_df.groupby('Date')[['COGS', 'Revenue']].sum().reset_index()

# Tạo biểu đồ vùng với chế độ không cộng dồn
fig = px.area(daily_cogs_revenue, 
              x='Date', 
              y=['COGS', 'Revenue'], 
              title='Phân tích Lợi nhuận gộp: Doanh thu vs Giá vốn',
              color_discrete_map={'Revenue': 'blue', 'COGS': 'red'})

# Chỉnh độ trong suốt (opacity) để nhìn thấy phần giao nhau
fig.update_traces(opacity=0.5)
fig.update_layout(barmode='overlay', hovermode='x unified')
fig.show()

#

In [ ]:
# 2. Sản phẩm & khách hàng**
# Load dữ liệu sản phẩm và khách hàng
products_df = pd.read_csv(DATA_DIR + 'products.csv')
customers_df = pd.read_csv(DATA_DIR + 'customers.csv')
orders_item_df = pd.read_csv(DATA_DIR + 'order_items.csv')
orders_df = pd.read_csv(DATA_DIR + 'orders.csv')
# Vẽ Top 5 sản phẩm được bán nhiều nhất theo category
# Tính tổng revenue trong orders_items sau đó merge với products để lấy category
orders_item_df['Revenue'] = orders_item_df['quantity'] * orders_item_df['unit_price']
product_revenue = orders_item_df.groupby('product_id')['Revenue'].sum().reset_index()
product_revenue = product_revenue.merge(products_df[['product_id', 'category']], on='product_id', how='left')
category_revenue = product_revenue.groupby('category')['Revenue'].sum().reset_index()
fig = px.bar(category_revenue, x='category', y='Revenue', title='Doanh thu theo category sản phẩm')
fig.update_layout(xaxis_title='Category', yaxis_title='Doanh thu (VND)')
fig.show()

# Phân bố segment theo đơn hàng/ số lượng sản phẩm
# Biểu đồ sử dụng: Pie chart
# merge cột segment từ products vào orders_item để đếm số lượng đơn hàng theo segment
orders_item_df = orders_item_df.merge(products_df[['product_id', 'segment']], on='product_id', how='left')
segment_counts_by_order = orders_item_df['segment'].value_counts().reset_index()
segment_counts_by_order.columns= ['segment', 'count']

# Phân bố segment theo quantity sản phẩm
segment_quantity = orders_item_df.groupby('segment')['quantity'].sum().reset_index()

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]], subplot_titles=['Số lượng đơn hàng theo segment', 'Số lượng sản phẩm theo segment'])
fig.add_trace(px.pie(segment_counts_by_order, values='count', names='segment').data[0], row=1, col=1)
fig.add_trace(px.pie(segment_quantity, values='quantity', names='segment').data[0], row=1, col=2)
fig.update_traces(textposition='inside', textinfo='percent+label') 
fig.update_layout(title_text='Phân khúc thị trường theo segment', showlegend=False)
fig.show()

# Phân bố size
# merge cột size từ products vào orders_item để đếm số lượng đơn hàng theo size
orders_item_df = orders_item_df.merge(products_df[['product_id', 'size']], on='product_id', how='left')
size_counts_by_order = orders_item_df['size'].value_counts().reset_index() 
size_counts_by_order.columns= ['size', 'count']
# Đếm số lượng sản phẩm theo size
size_quantity = orders_item_df.groupby('size')['quantity'].sum().reset_index()

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]], subplot_titles=['Số lượng đơn hàng theo size', 'Số lượng sản phẩm theo size'])
fig.add_trace(px.pie(size_counts_by_order, values='count', names='size').data[0], row=1, col=1)
fig.add_trace(px.pie(size_quantity, values='quantity', names='size').data[0], row=1, col=2)
fig.update_traces(textposition='inside', textinfo='percent+label') 
fig.update_layout(title_text='Phân bố size sản phẩm', showlegend=False)
fig.show()

#Số KH theo age_group (bar chart)
age_group_counts = customers_df['age_group'].value_counts().reset_index()
age_group_counts.columns = ['age_group', 'count']
fig = px.bar(age_group_counts, x='age_group', y='count', title='Số lượng khách hàng theo nhóm tuổi')
fig.update_layout(xaxis_title='Nhóm tuổi', yaxis_title='Số lượng khách hàng')
fig.show()

# Tổng số đơn hàng theo nhóm tuổi (bar chart) kèm theo đường line chỉ số đơn hàng trung bình theo nhóm tuổi (line chart)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Chuẩn bị dữ liệu
# Nối bảng để lấy thông tin nhóm tuổi
df_merged = orders_df.merge(customers_df[['customer_id', 'age_group']], on='customer_id', how='left')
df_merged = df_merged[df_merged['age_group'].notnull()] # Loại bỏ giá trị null theo yêu cầu Q6 

# Tính tổng đơn hàng và số khách hàng duy nhất mỗi nhóm
age_stats = df_merged.groupby('age_group').agg(
    total_orders=('order_id', 'nunique'),
    unique_customers=('customer_id', 'nunique')
).reset_index()

# Tính trung bình (Số đơn / Số khách) - Đây là chỉ số quan trọng cho Q6 [cite: 170]
age_stats['avg_orders'] = age_stats['total_orders'] / age_stats['unique_customers']
age_stats = age_stats.sort_values('age_group') # Đảm bảo thứ tự nhóm tuổi đúng

# 2. Vẽ biểu đồ Dual Axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Thêm Bar Chart (Tổng đơn hàng)
fig.add_trace(
    go.Bar(x=age_stats['age_group'], y=age_stats['total_orders'], name="Tổng số đơn hàng"),
    secondary_y=False,
)

# Thêm Line Chart (Trung bình mỗi khách)
fig.add_trace(
    go.Scatter(x=age_stats['age_group'], y=age_stats['avg_orders'], name="Đơn hàng TB/Khách", mode='lines+markers'),
    secondary_y=True,
)

fig.update_layout(title_text='Phân tích hành vi mua sắm theo Nhóm tuổi')
fig.update_xaxes(title_text='Nhóm tuổi')
fig.update_yaxes(title_text='Tổng số đơn hàng (Quy mô)', secondary_y=False)
fig.update_yaxes(title_text='Số đơn/Khách (Lòng trung thành)', secondary_y=True)
fig.show()

# Doanh thu theo nhóm tuổi (bar chart) kèm theo đường line chỉ số doanh thu trung bình theo nhóm tuổi (line chart)
# Tính doanh thu theo nhóm tuổi cần merge thêm cột Revenue có sẵn từ orders_item_df vào df_merged
df_merged = df_merged.merge(orders_item_df[['order_id', 'Revenue']], on='order_id', how='left')
age_revenue_stats = df_merged.groupby('age_group').agg(
    total_revenue=('Revenue', 'sum'),
    avg_revenue=('Revenue', 'mean')
).reset_index()
age_revenue_stats = age_revenue_stats.sort_values('age_group') # Đảm bảo thứ tự nhóm tuổi đúng
fig = make_subplots(specs=[[{"secondary_y": True}]])
# Thêm Bar Chart (Tổng doanh thu)
fig.add_trace(
    go.Bar(x=age_revenue_stats['age_group'], y=age_revenue_stats['total_revenue'], name="Tổng doanh thu"),
    secondary_y=False,
)
# Thêm Line Chart (Doanh thu trung bình mỗi khách)
fig.add_trace(
    go.Scatter(x=age_revenue_stats['age_group'], y=age_revenue_stats['avg_revenue'], name="Doanh thu TB/Khách", mode='lines+markers'),
    secondary_y=True,
)  
fig.update_layout(title_text='Phân tích doanh thu theo Nhóm tuổi')
fig.update_xaxes(title_text='Nhóm tuổi')
fig.update_yaxes(title_text='Tổng doanh thu (Quy mô)', secondary_y=False)
fig.update_yaxes(title_text='Doanh thu/Khách (Lòng trung thành)', secondary_y=True)
fig.show()


### Phân tích Cohort theo năm

In [ ]:
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
sales_df['Date'] = pd.to_datetime(sales_df['Date'])

master_df = pd.merge(orders_df, customers_df, on='customer_id', how='left')
master_df = pd.merge(master_df, sales_df, left_on='order_date', right_on='Date', how='left')

# 1. Định nghĩa năm mua hàng
master_df['order_year'] = master_df['order_date'].dt.year
master_df['cohort_year'] = master_df.groupby('customer_id')['order_date'].transform('min').dt.year

# 2. Tính số năm sau lần mua đầu (Index theo năm)
master_df['cohort_index_year'] = master_df['order_year'] - master_df['cohort_year'] + 1

# 3. Tạo bảng ma trận Cohort theo Năm
cohort_data_year = master_df.groupby(['cohort_year', 'cohort_index_year'])['customer_id'].nunique().reset_index()
cohort_matrix_year = cohort_data_year.pivot(index='cohort_year', columns='cohort_index_year', values='customer_id')

# 4. Tính tỷ lệ giữ chân (%)
cohort_size_year = cohort_matrix_year.iloc[:, 0]
retention_year = cohort_matrix_year.divide(cohort_size_year, axis=0)

# Vẽ biểu đồ Heatmap
fig = px.imshow(retention_year.astype(float),
                text_auto='.1%',
                color_continuous_scale='Blues',
                labels=dict(x="Số năm sau lần mua đầu", y="Năm gia nhập", color="Tỷ lệ giữ chân"),
                title="Tỷ lệ giữ chân khách hàng theo Năm (2012 - 2022)")
fig.show()

**Nhận xét**
- Nhóm khách hàng trung thành (2012 - 2013): Tỷ lệ giữ chân sau hơn 10 năm vẫn duy trì ở mức cao (~43.6%). Điều này cho thấy giai đoạn đầu doanh nghiệp đã xây dựng được một tệp khách hàng cực kỳ trung thành.
- Xu hướng sụt giảm nghiêm trọng: Tỷ lệ quay lại vào năm thứ hai (N+1) giảm dốc đứng theo thời gian. Nhóm gia nhập năm 2012 có 64.7% khách quay lại, nhưng nhóm năm 2021 chỉ còn lại 6.7%.
- Kết luận: Doanh nghiệp đang gặp vấn đề lớn trong việc thuyết phục khách hàng mới ở lại. Hiệu quả của các chiến dịch Marketing tìm kiếm khách hàng mới đang thấp dần về mặt chất lượng.

### Phân tích Cohort theo nhóm tuổi

In [ ]:
# 1. Group theo độ tuổi và index tháng
age_cohort = master_df.groupby(['age_group', 'cohort_index_year'])['customer_id'].nunique().reset_index()
age_matrix = age_cohort.pivot(index='age_group', columns='cohort_index_year', values='customer_id')

# 2. Tính tỷ lệ % và điền 0 cho các ô không có khách quay lại
age_retention = age_matrix.divide(age_matrix.iloc[:, 0], axis=0).fillna(0)

# 3. Vẽ biểu đồ - Chỉ xem 24 tháng đầu để tránh bị quá dài ngang
# Cắt bớt cột để biểu đồ tập trung vào 2 năm đầu tiên
fig = px.imshow(age_retention.iloc[:, :24].astype(float), 
                text_auto='.1%',
                color_continuous_scale='GnBu',
                labels=dict(x="Số năm sau lần mua đầu", y="Nhóm tuổi", color="Tỷ lệ giữ chân"),
                title="Lòng trung thành của khách hàng theo Độ tuổi")

fig.update_layout(xaxis_tickmode='linear')
fig.show()

**Nhận xét**
- Sự đồng nhất tuyệt đối: Dữ liệu cho thấy một kịch bản bất ngờ: tất cả các nhóm tuổi từ 18-24 đến 55+ đều có hành vi giữ chân gần như y hệt nhau. Tất cả đều giảm từ 100% xuống ~40% ở năm thứ hai và kết thúc ở mức ~10% sau 11 năm.
- Insight cốt lõi: Sản phẩm có tính đại chúng rất cao. Việc khách hàng rời đi không phải do lỗi "nhắm sai đối tượng mục tiêu", mà là một vấn đề ảnh hưởng đến toàn bộ trải nghiệm người dùng.

**Đề xuất chiến lược**
- Retention Campaign cho năm thứ 2: Vì 60% khách hàng của tất cả lứa tuổi đều bỏ đi sau năm đầu tiên, đây là giai đoạn "vàng" cần tung ra các chương trình ưu đãi tri ân để giữ chân họ.
- Rà soát chất lượng sản phẩm: So sánh danh mục sản phẩm chủ lực của giai đoạn 2012-2015 với hiện tại để tìm ra sự khác biệt về giá trị cốt lõi.

### Phân tích Revenue và COGS tháng 8 hàng năm

In [ ]:
df_diag = pd.merge(orders_df, sales_df, left_on='order_date', right_on='Date', how='inner')

# 4. Tính toán Gross Profit Margin
df_diag['Gross_Profit'] = df_diag['Revenue'] - df_diag['COGS']
df_diag['GP_Margin'] = (df_diag['Gross_Profit'] / df_diag['Revenue']) * 100

# 5. Lọc riêng dữ liệu tháng 8 của tất cả các năm
august_data = df_diag[df_diag['order_date'].dt.month == 8]

# 6. Tổng hợp dữ liệu theo năm để tìm thủ phạm
aug_summary = august_data.groupby(august_data['order_date'].dt.year).agg({
    'Revenue': 'sum',
    'COGS': 'sum',
    'GP_Margin': 'mean'
}).reset_index()

# 7. Trực quan hóa để so sánh doanh thu và chi phí vón
fig = px.bar(aug_summary, x='order_date', y=['Revenue', 'COGS'], 
             barmode='group',
             title="Chẩn đoán tháng 8: Doanh thu vs Chi phí vốn (2012-2022)",
             labels={'order_date': 'Năm', 'value': 'Số tiền (VND)', 'variable': 'Chỉ số'})

fig.show()

print("Chi tiết biên lợi nhuận tháng 8 qua các năm:")
print(aug_summary[['order_date', 'GP_Margin']].sort_values(by='order_date'))

$\Rightarrow$ Ta thấy cứ mỗi tháng 8 vào năm lẻ 2013, 2015 thì biên lợi nhuận đều âm cho thấy có tính chu kì xảy ra ở đây

In [ ]:
# So sánh các chỉ số trung bình của tháng 8 năm Lẻ (Lỗ) vs Năm Chẵn (Lời)
aug_comparison = august_data.copy()
aug_comparison['Year_Type'] = aug_comparison['order_date'].dt.year.apply(lambda x: 'Lẻ (Lỗ)' if x % 2 != 0 else 'Chẵn (Lời)')

final_diag = aug_comparison.groupby('Year_Type').agg({
    'order_id': 'count',           # Tổng số đơn
    'Revenue': ['sum', 'mean'],     # Tổng doanh thu & Doanh thu trung bình mỗi ngày
    'COGS': ['sum', 'mean']         # Tổng vốn & Vốn trung bình mỗi ngày
}).round(0)

print("--- So sánh quy mô doanh số tháng 8 ---")
final_diag

$\Rightarrow$ Chỉ số COGS ở năm chẵn hay lẻ gần như không đổi duy trì ở mức 5.5 - 5.6 triệu/đơn nhưng Revenue trung bình ở các năm chăn là 7 triệu nhưng năm lẻ lại chỉ còn 4.2 triệu/đơn

In [ ]:
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')

# 2. Xử lý định dạng ngày tháng trong bảng orders
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

# 1. Liên kết sản phẩm với chi tiết đơn hàng để có trường 'category'
df_items_with_cat = pd.merge(
    order_items, 
    products_df[['product_id', 'category', 'price', 'cogs']], 
    on='product_id', 
    how='left'
)

df_full = pd.merge(
    df_items_with_cat, 
    orders_df[['order_id', 'order_date']], 
    on='order_id', 
    how='left'
)

df_full['order_date'] = pd.to_datetime(df_full['order_date'])

# 3. Lọc dữ liệu tháng 8 của các năm lẻ
target_years = [2013, 2015, 2017, 2019, 2021]
aug_loss_details = df_full[
    (df_full['order_date'].dt.month == 8) & 
    (df_full['order_date'].dt.year.isin(target_years))
].copy() # Dùng copy() để tránh cảnh báo SettingWithCopy

# 4. Đổi tên cột cho đồng bộ với logic tính toán của bạn (nếu cần)
aug_loss_details = aug_loss_details.rename(columns={
    'price': 'Revenue',
    'cogs': 'COGS'
})

# 5. Bây giờ cột 'Revenue' và 'COGS' đã tồn tại, bạn có thể Groupby bình thường
cat_summary = aug_loss_details.groupby('category').agg({
    'Revenue': 'sum',
    'COGS': 'sum'
}).reset_index()

# 6. Tính toán % Margin và vẽ biểu đồ
cat_summary['GP_Margin'] = (cat_summary['Revenue'] - cat_summary['COGS']) / cat_summary['Revenue'] * 100
cat_summary = cat_summary.sort_values(by='GP_Margin')

fig = px.bar(cat_summary, x='category', y='GP_Margin', 
             color='GP_Margin', color_continuous_scale='RdBu',
             title="Biên lợi nhuận theo Ngành hàng (Tháng 8 năm lẻ)",
             labels={'GP_Margin': 'Biên lợi nhuận (%)', 'category': 'Ngành hàng'})
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.show()

**Nhận xét**
Dựa vào các ngành hàng thì biên lợi nhuận vẫn dương nhưng đây là do đang tính toán dự trên giá niêm yết -> về bản chất thì đều có biên lợi nhuận gốc rất tốt (17-23%)

$\Rightarrow$ Có thể nhận định nguyên nhân gây lỗ ở mốc thời gian này là do chiến dịch khuyến mãi sâu làm làm cho biên lợi nhuận âm rất nhiều.

**Đề xuất giải pháp**
- Kiểm soát trần chiết khấu (Discount Ceiling): Tuyệt đối không để tổng mức giảm giá vượt quá biên lợi nhuận gốc của sản phẩm (ví dụ: nhóm Casual không được giảm quá 15%).

- Thay đổi cách tiếp cận tháng 8: Thay vì chạy đua giảm giá diện rộng vốn đã chứng minh là không hiệu quả, nên chuyển sang các chiến dịch Targeted Marketing nhắm vào nhóm khách hàng cũ hoặc các combo sản phẩm có lãi suất cao.

- Rà soát danh mục: Ưu tiên đẩy mạnh các ngành hàng có biên lãi gốc cao như GenZ và Outdoor (>22%) để có dư địa khuyến mãi an toàn hơn.